# SGLang: Fast Structured Generation and Serving

## What is SGLang?

**SGLang** (Structured Generation Language) is an inference framework that surpassed vLLM in 2024-2025 
on most throughput benchmarks. It's now powering production workloads at xAI, NVIDIA, AMD, and major cloud providers
(reportedly ~400,000 GPUs as of early 2025).

**Key innovations over vLLM:**
- **RadixAttention**: smarter prefix caching using a radix tree — shares KV cache across requests with common prefixes
- **Zero-overhead CPU scheduler**: continuous batching with lower latency than vLLM's Python scheduler
- **Disaggregated prefill/decode**: separate servers for the prefill phase vs decode phase (halves latency for long prompts)
- **Native multi-LoRA batching**: serve multiple LoRA adapters simultaneously with minimal overhead

**Installation note:** SGLang requires CUDA 12.1+ (check your environment).

In [ ]:
import subprocess
result = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
print(result.stdout)

import torch
print(f"CUDA {torch.version.cuda} | Compute {torch.cuda.get_device_capability()}")

# Check if CUDA >= 12.1 for SGLang
major, minor = torch.version.cuda.split(".")[:2]
if int(major) >= 12 and int(minor) >= 1:
    print("✓ CUDA 12.1+ — SGLang supported")
else:
    print(f"⚠ CUDA {torch.version.cuda} — may have issues, SGLang recommends 12.1+")

In [ ]:
!pip install sglang -q 2>&1 | tail -5

In [ ]:
# Test SGLang offline engine (no server needed for basic inference)
import sglang as sgl
import torch, gc

gc.collect(); torch.cuda.empty_cache()
print(f"SGLang {sgl.__version__}")

# Initialize SGLang's offline engine
llm = sgl.Engine(
    model_path="Qwen/Qwen3-0.6B",
    dtype="float16",
    mem_fraction_static=0.7,  # fraction of GPU memory for KV cache
)

# Batch generation — SGLang handles continuous batching automatically
prompts = [
    "What is LoRA fine-tuning? Answer in one sentence.",
    "Explain gradient descent briefly.",
    "What is the key advantage of vLLM over standard transformers?",
    "Define quantization in the context of LLMs.",
]

sampling_params = {"temperature": 0.7, "max_new_tokens": 80}
outputs = llm.generate(prompts, sampling_params)

print("=== SGLang Generation Results ===")
for prompt, output in zip(prompts, outputs):
    print(f"\nQ: {prompt}")
    print(f"A: {output['text'].strip()}")

# Throughput measurement
import time
big_batch = prompts * 8  # 32 prompts
t0 = time.time()
outputs_big = llm.generate(big_batch, sampling_params)
elapsed = time.time() - t0
total_tokens = sum(len(o["text"].split()) for o in outputs_big)
print(f"\n=== Throughput ===")
print(f"  {len(big_batch)} prompts in {elapsed:.2f}s")
print(f"  ~{total_tokens/elapsed:.0f} tokens/sec")

## SGLang vs vLLM: When to Use Each

| Feature | SGLang | vLLM |
|---------|--------|------|
| Throughput | Generally higher (+20-40%) | Good baseline |
| Prefix caching | RadixAttention (smarter) | Standard prefix cache |
| Prefill latency | Can disaggregate | Single server |
| Multi-LoRA | Yes | Yes |
| OpenAI API | Yes | Yes |
| Ease of setup | Similar | Slightly simpler |
| Community | Growing fast | Larger ecosystem |
| Best for | High-throughput production | General use, simpler setup |

## SGLang as OpenAI-Compatible Server

```bash
# Launch SGLang as drop-in OpenAI API replacement
python -m sglang.launch_server \
    --model-path Qwen/Qwen3-0.6B \
    --host 0.0.0.0 \
    --port 8000 \
    --dtype float16

# Then use with OpenAI client
from openai import OpenAI
client = OpenAI(base_url="http://localhost:8000/v1", api_key="none")
response = client.chat.completions.create(
    model="Qwen/Qwen3-0.6B",
    messages=[{"role": "user", "content": "What is LoRA?"}],
    max_tokens=200,
)
```